# GA4 Data Extraction for Multiple Websites

**Goal:**
This notebook automates the process of fetching key performance metrics from the Google Analytics 4 (GA4) Data API for a list of specified websites.

**Process:**
1.  **Authentication:** Securely connects to the Google Analytics API using a service account.
2.  **Configuration:** Loads a list of website property IDs from an external JSON file.
3.  **Data Fetching:** Iterates through each website ID, sending a request to the API for a predefined set of dimensions and metrics within a specified date range.
4.  **Output:** For each website, the raw data is processed from the API response object into a clean list of dictionaries and saved as a separate JSON file in a designated output directory.

**Final Output:** A collection of JSON files, each named after its corresponding website, containing the raw analytics data.   

**Required libraries**

In [1]:
!pip install google-analytics-data

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 3.3 MB/s eta 0:00:00


In [2]:
# Save files
import json
import os

# Google Analytics Modules and Libraries
from google.oauth2 import service_account
from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.analytics.data_v1beta.types import (
    DateRange,
    Dimension,
    Metric,
    RunReportRequest,
    OrderBy,
)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Access data to API and our websites IDs**

In [4]:
# Input paths
KEY_FILE_PATH = '/content/drive/MyDrive/01_data_science/03_projects/02_API_GA/1_access_data/data-project-438907-2e1111c8bd46.json'
WEBSITES_ID_PATH = '/content/drive/MyDrive/01_data_science/03_projects/02_API_GA/1_access_data/websites_id.json'

# Save path
SAVE_PATH_DIR = '/content/drive/MyDrive/01_data_science/03_projects/02_API_GA/3_data/raw'

In [5]:
# Path to the login file (JSON service account key)
key_file = KEY_FILE_PATH

# Property ID (22 websites)
with open(WEBSITES_ID_PATH, 'r', encoding='utf-8') as json_file:
    websites_id = json.load(json_file)

# Authentication by service account
scopes = ['https://www.googleapis.com/auth/analytics.readonly']
credentials = service_account.Credentials.from_service_account_file(key_file, scopes=scopes)

# Service API initialization
client = BetaAnalyticsDataClient(credentials=credentials)

**Input request parameters**  

In [6]:
# Specify the date range
start_date = '2023-01-01'
end_date = '2023-12-31'
date_range = [
    DateRange(
    start_date=start_date,
    end_date=end_date
)]

# Specify the dimensions
dimensions = [
    Dimension(name='year'),
    Dimension(name='month'),
    Dimension(name='deviceCategory'),
    Dimension(name='sessionDefaultChannelGroup'),
]

# Specify the metrics
metrics = [
    Metric(name='activeUsers'), # The number of distinct users who visited website during a specified time period
    Metric(name='newUsers'), # The number of users who interacted with website for the first time during a specified time period
    Metric(name='sessions'), # The total number of individual sessions initiated by users during a specified time period
    Metric(name='sessionsPerUser'), # The average number of sessions per user within the specified time period
    Metric(name='screenPageViews'), # The number of page views during a specified time period (repeated views of a single page are counted)
    Metric(name='engagedSessions'), # The number of sessions that lasted longer than 10 seconds, or had a key event, or had 2 or more screen views
    Metric(name='averageSessionDuration'), # The average time users spend in a session, measured in seconds
    Metric(name='userEngagementDuration'), # Total time users actively interacted with the page in seconds
]

# Specify the order in which the data will be returned.
order_by = [
    OrderBy(dimension={'dimension_name': 'year'}, desc=False),
    OrderBy(dimension={'dimension_name': 'month'}, desc=False),
]

**Data extraction**

*1) Functions definition*

In [7]:
def get_report(client, id):
    """Create the request with the requests parameters.
    Send the request to the API"""
    request = RunReportRequest(
        property=f'properties/{id}',
        date_ranges=date_range,
        dimensions=dimensions,
        metrics=metrics,
        order_bys=order_by,
    )

    response = client.run_report(request)
    return response

In [8]:
def check_response(response):
    """Checking the content of API response object"""
    print('The response rowcount: ', response.row_count, '\n')

    print('The response dimension headers:')
    for header in response.dimension_headers:
        print(' ', header.name)

    print('\nThe API response metric headers:')
    for header in response.metric_headers:
        print(' ', header.name)

    print('\nSample data rows:')
    for row in response.rows[:5]:
        dimensions = [dim.value for dim in row.dimension_values]
        metrics = [metric.value for metric in row.metric_values]
        print(' Dimensions:', dimensions, ' Metrics:', metrics)

In [9]:
def process_response(response):
    """Process the API response to extract the data and return a list of dictionaries."""
    dim_len = len(response.dimension_headers)
    metric_len = len(response.metric_headers)

    raw_data = []

    # Process each row in the response
    for row in response.rows:
        row_data = {}
        # Extract dimension values
        for i in range(dim_len):
            dim_name = response.dimension_headers[i].name
            dim_value = row.dimension_values[i].value
            row_data[dim_name] = dim_value
        # Extract metric values
        for i in range(metric_len):
            metric_name = response.metric_headers[i].name
            metric_value = row.metric_values[i].value
            row_data[metric_name] = metric_value

        raw_data.append(row_data)

    return raw_data

In [10]:
def save_raw_data(raw_data, file_path):
    """Saves the row data to JSON file"""
    with open(file_path, mode='w', encoding='utf-8') as file:
        json.dump(raw_data, file, default=str, indent=4)

In [11]:
def check_file_saved(file_path):
    """Check if the file was saved successfully."""
    if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
        print(f'File {file_path} saved successfully.')
    else:
        print(f'Error saving file: {file_path}')

*2) Getting data from all our websites*

In [12]:
for key, value in websites_id.items():
    try:
        print(f'\nProcessing website: {key} (ID: {value})')
        # Get API response
        response = get_report(client, value)

        # Check response
        check_response(response)

        # Response processing
        raw_data = process_response(response)

        # Skip saving if no data was returned
        if not raw_data:
            print(f'--> No data returned for {key}. Skipping file save.')
            continue

        # Save to JSON
        file_name = f'{key}-{start_date}-{end_date}.json'
        full_save_path = os.path.join(SAVE_PATH_DIR, file_name)
        save_raw_data(raw_data, full_save_path)

        # File save check
        check_file_saved(full_save_path)

    except Exception as e:
        # If any error occurs for one website, print it and continue with the next one
        print(f'!!! An error occurred for website {key} !!!')
        print(f'Error details: {e}')
        continue # Move to the next website


Processing website: denzemedelce.cz (ID: 357740205)
The response rowcount:  33 

The response dimension headers:
  year
  month
  deviceCategory
  sessionDefaultChannelGroup

The API response metric headers:
  activeUsers
  newUsers
  sessions
  sessionsPerUser
  screenPageViews
  engagedSessions
  averageSessionDuration
  userEngagementDuration

Sample data rows:
 Dimensions: ['2023', '03', 'desktop', 'Direct']  Metrics: ['1', '1', '1', '1', '2', '1', '110.968225', '108']
 Dimensions: ['2023', '03', 'mobile', 'Direct']  Metrics: ['1', '1', '1', '1', '1', '1', '63.169688', '40']
 Dimensions: ['2023', '04', 'mobile', 'Direct']  Metrics: ['4', '4', '4', '1', '4', '3', '48.94822225', '193']
 Dimensions: ['2023', '05', 'desktop', 'Direct']  Metrics: ['4', '4', '4', '1', '7', '4', '225.8262955', '439']
 Dimensions: ['2023', '05', 'mobile', 'Direct']  Metrics: ['1', '1', '1', '1', '1', '1', '51.111815', '46']
File /content/drive/MyDrive/01_data_science/03_projects/02_API_GA/3_data/raw/denze